# Notebook 04 — Exploratory Analysis

This notebook runs all five research questions and explores the results interactively.

> Prerequisite: Run Notebook 01 and Notebook 03 first (or run `python src/sample_data_generator.py && python src/clean_data.py`).

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from analysis import load_all_clean, q1_before_after_comparison, q2_platform_spike, \
    q3_event_spike_ratios, q4_top_sources, q5_topic_dominance
from config import WAR_START, WAR_EVENTS

data = load_all_clean()
print(f"Loaded datasets: {list(data.keys())}")

## Q1: Did news activity increase after October 7?

In [ ]:
q1 = q1_before_after_comparison(data)
q1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(q1))
width = 0.35
bars1 = ax.bar([i - width/2 for i in x], q1['before_mean'], width, label='Before Oct 7', color='#aec7e8')
bars2 = ax.bar([i + width/2 for i in x], q1['after_mean'], width, label='After Oct 7', color='#ff7f0e')
ax.set_xticks(list(x))
ax.set_xticklabels(q1['metric'], rotation=20, ha='right')
ax.set_title('Before vs. After October 7: Average Daily Activity')
ax.legend()
plt.tight_layout()
plt.show()

## Q2: Which platform showed the strongest spike?

In [ ]:
q2 = q2_platform_spike(data)
q2

## Q3: Which events produced the strongest reaction?

In [ ]:
q3 = q3_event_spike_ratios(data)
q3

In [ ]:
pivot = q3.pivot_table(index='event_name', columns='platform', values='spike_ratio')
pivot.plot.bar(figsize=(10, 5), rot=20, title='Spike Ratio by Event and Platform')
plt.axhline(1.0, color='black', linestyle='--', alpha=0.5)
plt.ylabel('Spike Ratio (event window / baseline)')
plt.tight_layout()
plt.show()

## Q4: Which sources were most active?

In [ ]:
q4 = q4_top_sources(data)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if 'telegram' in q4:
    q4['telegram'].set_index('source_name')['total_messages'].sort_values().plot.barh(
        ax=axes[0], color='#229ED9', alpha=0.8, title='Telegram: Total Messages by Channel')

if 'rss' in q4:
    q4['rss'].set_index('source_name')['total_articles'].sort_values().plot.barh(
        ax=axes[1], color='#E63946', alpha=0.8, title='RSS: Total Articles by Source')

plt.tight_layout()
plt.show()

## Q5: Which topics became more dominant?

In [ ]:
q5 = q5_topic_dominance(data)
q5.head(12)

In [ ]:
rss_topics = q5[q5['source'] == 'RSS'].dropna(subset=['growth_ratio'])
rss_topics.set_index('topic')['growth_ratio'].sort_values().plot.barh(
    figsize=(9, 4), color='#E63946', alpha=0.8, title='RSS Topic Growth Ratio (after/before Oct 7)')
plt.axvline(1.0, color='black', linestyle='--', alpha=0.5)
plt.xlabel('Growth ratio')
plt.tight_layout()
plt.show()

## Key Observations

*(Based on simulated data — not real-world evidence)*

1. All platforms show increased activity after October 7.
2. Telegram shows the sharpest spike ratio, consistent with its role as a real-time alert platform.
3. The October 7 event itself produces the largest spike; subsequent events show smaller but measurable bumps.
4. RSS article volume increase reflects increased publication output from news organizations.
5. Google Trends keywords like 'hostages' and 'siren' show the highest relative growth.

Proceed to **Notebook 05** for final portfolio-ready charts.